# 05 - Data Lineage Visualization

Demonstrate data lineage tracking across lakehouse layers.
Shows how data can be traced from Gold back to Bronze source files.

In [ ]:
import sys
sys.path.insert(0, '/opt/spark-jobs')
from utils.spark_session import create_spark_session
from utils.delta_helpers import read_delta, get_table_history
from pyspark.sql import functions as F

spark = create_spark_session(app_name='LineageDemo')

In [ ]:
# Lineage: Trace a single customer through all layers
CUSTOMER_ID = 'CUST-000001'

print(f'=== Lineage Trace for {CUSTOMER_ID} ===')
print()

# Bronze layer
bronze_cust = read_delta(spark, 's3a://bronze/customers').filter(
    F.col('customer_id') == CUSTOMER_ID
)
print('📥 BRONZE (Raw):')
bronze_cust.select('customer_id', 'first_name', 'last_name',
                    '_ingestion_timestamp', '_source_file').show(truncate=False)

In [ ]:
# Silver layer
silver_cust = read_delta(spark, 's3a://silver/customers_cleansed').filter(
    F.col('customer_id') == CUSTOMER_ID
)
print('🔄 SILVER (Cleansed):')
silver_cust.select('customer_id', 'first_name', 'last_name',
                    'risk_category', '_cleansed_timestamp').show(truncate=False)

In [ ]:
# Gold layer
gold_cust = read_delta(spark, 's3a://gold/customer_360').filter(
    F.col('customer_id') == CUSTOMER_ID
)
print('📊 GOLD (Customer 360):')
gold_cust.select('customer_id', 'customer_name', 'risk_category',
                  'total_accounts', 'total_transactions',
                  'aml_alert_count', '_gold_timestamp').show(truncate=False)

In [ ]:
# Delta Lake version history per layer
print('=== Delta Lake Version History ===')
for layer, path in [
    ('Bronze', 's3a://bronze/customers'),
    ('Silver', 's3a://silver/customers_cleansed'),
    ('Gold',   's3a://gold/customer_360'),
]:
    try:
        history = get_table_history(spark, path, limit=3)
        print(f'\n📋 {layer} History:')
        history.select('version', 'timestamp', 'operation',
                       'operationMetrics').show(truncate=False)
    except Exception as e:
        print(f'\n⚠️ {layer}: Not available ({e})')

In [ ]:
# Lineage summary visualization (text-based)
print('''
╔══════════════════════════════════════════════════════════════════╗
║                    DATA LINEAGE MAP                             ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║   📂 Source Files (CSV/JSON)                                     ║
║       │                                                          ║
║       ▼                                                          ║
║   🥉 BRONZE (Delta Lake)                                        ║
║       │  + _ingestion_timestamp                                  ║
║       │  + _source_file                                          ║
║       │  + _data_classification                                  ║
║       │  + _data_residency                                       ║
║       ▼                                                          ║
║   🥈 SILVER (Delta Lake)                                        ║
║       │  + Deduplication                                         ║
║       │  + Type casting                                          ║
║       │  + AML flagging                                          ║
║       │  + _cleansed_timestamp                                   ║
║       ▼                                                          ║
║   🥇 GOLD (Delta Lake)                                          ║
║       │  Customer 360 (joined view)                              ║
║       │  AML Summary (aggregated)                                ║
║       │  Regulatory Report (hashed, CBUAE-ready)                 ║
║       ▼                                                          ║
║   📊 Analytics & Reporting                                      ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
''')

In [ ]:
spark.stop()